In [8]:
import os
import re

def translate(input_file, output_file, bsim_level=54):
   
    if not os.path.exists(input_file):
        print(f"Error: Source file '{input_file}' not found.")
        return

   
    valid_bsim4_parameters = {
        'vth0', 'vtho', 'k1', 'k2', 'k3', 'k3b', 'w0', 'dv0', 'dvt1', 'dvt2', 
        'dvt0w', 'dvt1w', 'dvt2w', 'xl', 'xw', 'vfb', 'delvto', 'eta0', 'etab',
        'u0', 'ua', 'ub', 'uc', 'vsat', 'a0', 'ags', 'a1', 'a2', 'b0', 'b1',
        'rdsw', 'rds0', 'prwg', 'prwb', 'wr', 'rsh', 'rd', 'rs',
        'nfactor', 'voff', 'cit', 'cdsc', 'cdscb', 'cdscd', 
        'toxe', 'toxm', 'toxp', 'xj', 'ndep', 'nsub', 'ngate', 'gamma1', 'gamma2',
        'alpha0', 'alpha1', 'beta0', 'pclm', 'pdiblbl1', 'pdiblbl2', 'pdiblcb',
        'cj', 'mj', 'pb', 'cjsw', 'mjsw', 'pbsw', 'cjswg', 'mjswg', 'pbswg',
        'cgdo', 'cgso', 'cgbo', 'cf', 'tbox', 'tempe'
    }

    with open(input_file, 'r') as f:
        lines = f.readlines()

    print(f"Processing {len(lines)} lines with clean structural constraints...")
    
    param_regex = re.compile(r'(?:\.param\s+)?([\w\d_]+)\s*=\s*([^\s]+)', re.IGNORECASE)
    
    translated_output = []
    translated_output.append("* --- Entirely Translated Master SPICE File --- \n")

    for line in lines:
        clean_line = line.strip()
        
        if not clean_line:
            translated_output.append("")
            continue
            
        if clean_line.startswith('*') or clean_line.startswith('#'):
            translated_output.append(line.rstrip())
            continue

        match = param_regex.search(clean_line)
        if match:
            raw_name = match.group(1).strip()
            param_value = match.group(2).strip().strip("'\"{}")
            base_name = raw_name.lower().split('_')[0]
            
            
            if base_name in valid_bsim4_parameters or raw_name.lower() in valid_bsim4_parameters:
                translated_output.append(f"+ {raw_name} = {param_value}")
            else:
                
                if raw_name.lower() in ['w', 'l', 'wmin', 'wmax', 'lmin', 'lmax']:
                    translated_output.append(f".param {raw_name} = {param_value}")
                else:
                    continue
        else:
           
            if clean_line.lower().startswith('.subckt'):
                translated_output.append("\n.subckt sky130_fd_pr__nfet_01v8_lvt d g s b")
                translated_output.append("+ .param l=1 w=1 nf=1.0 ad=0 as=0 pd=0 ps=0 nrd=0 nrs=0 sa=0 sb=0 sd=0 mult=1\n")
                translated_output.append("msky130_fd_pr__nfet_01v8_lvt d g s b sky130_fd_pr__nfet_01v8_lvt__model l={l} w={w} nf={nf} ad={ad} as={as} pd={pd} ps={ps} nrd={nrd} nrs={nrs} sa={sa} sb={sb} sd={sd} m={mult}\n")
                translated_output.append(".model sky130_fd_pr__nfet_01v8_lvt__model nmos")
                translated_output.append(f"+ level = {bsim_level}")
            elif clean_line.lower().startswith('.ends') or clean_line.lower() == '.end':
                translated_output.append(".ends sky130_fd_pr__nfet_01v8_lvt\n")
            else:
                translated_output.append(clean_line)

    with open(output_file, 'w') as out_f:
        out_f.write("\n".join(translated_output))
        
    print(f"Success! Output written to: {output_file}")


if __name__ == "__main__":
    translate("mystic_raw_output.txt", "sky130_nfet_sim_ready.spice")

Processing 4 lines with clean structural constraints...
Success! Output written to: sky130_nfet_sim_ready.spice


In [9]:

mystic = '/home/oliviag/sky130_fd_pr__nfet_01v8_lvt__tt_77k.corner.spice'
ngSpice = '/home/oliviag/TRANS_sky130_fd_pr__nfet_01v8_lvt__tt_77k.corner.spice'

translate(mystic, ngSpice, 54)

Processing 2582 lines with clean structural constraints...
Success! Output written to: /home/oliviag/TRANS_sky130_fd_pr__nfet_01v8_lvt__tt_77k.corner.spice
